# ISEF Engineering Project: Altamont Pass Autonomous Raptor Defense
## Google Colab Cloud Training Notebook: 4-Class YOLOv8-P2 Hard-Negative Model

This notebook trains an aerial detection model tailored for the Altamont Pass Wind Resource Area.
Target Taxonomy:
- **Class 0: Eagle_Protected** (Golden Eagle, Bald Eagle -> Triggers curtailment)
- **Class 1: Sympatric_Vulture** (Turkey Vulture -> Suppressed by taxonomic filter)
- **Class 2: Sympatric_Corvid_Buteo** (Common Raven, Red-tailed Hawk -> Suppressed)
- **Class 3: Anthropogenic_Airborne** (UAV / Aircraft / Paraglider -> Suppressed)

Architecture: **YOLOv8n-P2** with a stride-4 ($160 \times 160$) high-resolution detection head for distant raptors.

In [ ]:
# Step 1: Verify Hardware Accelerator
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics and dependencies
!pip install --upgrade ultralytics sahi onnx onnxruntime albumentations

In [ ]:
# Step 3: Write Dataset Configuration YAML
dataset_yaml = """
path: ./dataset
train: images/train
val: images/val
test: images/test

names:
  0: Eagle_Protected
  1: Sympatric_Vulture
  2: Sympatric_Corvid_Buteo
  3: Anthropogenic_Airborne
"""
with open('altamont_raptors.yaml', 'w') as f:
    f.write(dataset_yaml.strip())
print("Created altamont_raptors.yaml")

In [ ]:
# Step 4: Write YOLOv8n-P2 High-Resolution Architecture
p2_yaml = """
nc: 4
scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24, 27], 1, Detect, [nc]]
"""
with open('yolov8n-p2.yaml', 'w') as f:
    f.write(p2_yaml.strip())
print("Created yolov8n-p2.yaml")

In [ ]:
# Step 5: Transfer Learning & Training Execution
from ultralytics import YOLO

model = YOLO('yolov8n-p2.yaml').load('yolov8n.pt')
results = model.train(
    data='altamont_raptors.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    patience=20,
    project='altamont_training',
    name='yolov8n_p2_raptors',
    save=True
)

In [ ]:
# Step 6: Validate and Export ONNX Artifacts
val_metrics = model.val()
print(f"Validation mAP50: {val_metrics.box.map50:.4f}")
model.export(format='onnx', dynamic=True, simplify=True)
print("Exported best.pt and ONNX to Colab filesystem")